# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

# Lab 4 - Patent Join using PySpark DataFrames

The goal of this lab is to find the number of same-state citations for
each patent. A citation connects a citing patent to a cited patent. The
patent data contains the state associated with each patent.

For each citation, I need to find the state of both the citing patent and
the cited patent. I then keep only citations where both states are known
and are the same. Finally, I count these citations for each citing patent
and report the top 10.

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [3]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [4]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [5]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [6]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [8]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [9]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

## Prepare the patent data for the two lookups

The patents table is needed twice.

The first copy will be used to find the state of the citing patent.
The second copy will be used to find the state of the cited patent.

In [10]:
citing_patents = patents.select(
    col("PATENT").alias("CITING_PATENT"),
    col("POSTATE").alias("CITING_STATE")
)

In [11]:
citing_patents.show(5)

+-------------+------------+
|CITING_PATENT|CITING_STATE|
+-------------+------------+
|      3070801|        NULL|
|      3070802|          TX|
|      3070803|          IL|
|      3070804|          OH|
|      3070805|          CA|
+-------------+------------+
only showing top 5 rows



In [12]:
cited_patents = patents.select(
    col("PATENT").alias("CITED_PATENT"),
    col("POSTATE").alias("CITED_STATE")
)

In [13]:
cited_patents.show(5)

+------------+-----------+
|CITED_PATENT|CITED_STATE|
+------------+-----------+
|     3070801|       NULL|
|     3070802|         TX|
|     3070803|         IL|
|     3070804|         OH|
|     3070805|         CA|
+------------+-----------+
only showing top 5 rows



## Join the citation data with the patent states

I first join using the CITING patent number to obtain the state of the
patent making the citation.

I then join again using the CITED patent number to obtain the state of
the patent being cited.

In [14]:
citation_with_citing_state = citations.join(
    citing_patents,
    citations.CITING == citing_patents.CITING_PATENT,
    "left"
)

In [15]:
citation_with_citing_state.show(10)

+-------+-------+-------------+------------+
| CITING|  CITED|CITING_PATENT|CITING_STATE|
+-------+-------+-------------+------------+
|3858242|1515701|      3858242|          MI|
|3858242|3319261|      3858242|          MI|
|3858242|3668705|      3858242|          MI|
|3858242|3707004|      3858242|          MI|
|3858243|2949611|      3858243|        NULL|
|3858243|3146465|      3858243|        NULL|
|3858241| 956203|      3858241|          MA|
|3858241|1324234|      3858241|          MA|
|3858241|3398406|      3858241|          MA|
|3858241|3557384|      3858241|          MA|
+-------+-------+-------------+------------+
only showing top 10 rows



In [16]:
citation_with_states = citation_with_citing_state.join(
    cited_patents,
    citation_with_citing_state.CITED == cited_patents.CITED_PATENT,
    "left"
)

In [17]:
citation_with_states.show(10)

+-------+-------+-------------+------------+------------+-----------+
| CITING|  CITED|CITING_PATENT|CITING_STATE|CITED_PATENT|CITED_STATE|
+-------+-------+-------------+------------+------------+-----------+
|3858242|1515701|      3858242|          MI|        NULL|       NULL|
|3858242|3319261|      3858242|          MI|     3319261|         OH|
|3858241|3634889|      3858241|          MA|     3634889|         OH|
|3858241| 956203|      3858241|          MA|        NULL|       NULL|
|3858241|1324234|      3858241|          MA|        NULL|       NULL|
|3858243|2949611|      3858243|        NULL|        NULL|       NULL|
|3858243|3146465|      3858243|        NULL|     3146465|         MI|
|3858241|3398406|      3858241|          MA|     3398406|         FL|
|3858241|3557384|      3858241|          MA|     3557384|         MA|
|3858242|3668705|      3858242|          MI|     3668705|         WI|
+-------+-------+-------------+------------+------------+-----------+
only showing top 10 

## Filter same-state citations

A citation counts only when the citing patent and cited patent both have
state information and the two states are the same.

In [18]:
same_state = citation_with_states.filter(
    col("CITING_STATE").isNotNull() &
    col("CITED_STATE").isNotNull() &
    (col("CITING_STATE") == col("CITED_STATE"))
)

In [19]:
same_state.show(10)

+-------+-------+-------------+------------+------------+-----------+
| CITING|  CITED|CITING_PATENT|CITING_STATE|CITED_PATENT|CITED_STATE|
+-------+-------+-------------+------------+------------+-----------+
|4067198|3217791|      4067198|          AK|     3217791|         AK|
|4676695|3217791|      4676695|          AK|     3217791|         AK|
|5190098|3217791|      5190098|          AK|     3217791|         AK|
|5238053|3217791|      5238053|          AK|     3217791|         AK|
|4075779|3373523|      4075779|          AK|     3373523|         AK|
|4130086|3464385|      4130086|          AK|     3464385|         AK|
|4178878|3464385|      4178878|          AK|     3464385|         AK|
|4344414|3472314|      4344414|          AK|     3472314|         AK|
|5618134|3472314|      5618134|          AK|     3472314|         AK|
|4205718|3472314|      4205718|          AK|     3472314|         AK|
+-------+-------+-------------+------------+------------+-----------+
only showing top 10 

## Count same-state citations

Each remaining row represents one same-state citation.

I group the rows by the citing patent and count the rows in each group.

In [20]:
same_state_counts = same_state.groupBy("CITING").agg(
    count("*").alias("CO_CITED_COUNT")
)

In [21]:
same_state_counts.show(10)

+-------+--------------+
| CITING|CO_CITED_COUNT|
+-------+--------------+
|3968160|             1|
|5292488|             3|
|5179246|             1|
|5392941|             3|
|5734569|             2|
|4635660|             2|
|5037519|             5|
|4779427|             6|
|6001005|             2|
|5187901|             2|
+-------+--------------+
only showing top 10 rows



## Add the counts back to the patent data

The count is joined back to the original patent data so that the result
contains the original patent information together with CO_CITED_COUNT.

In [22]:
patent_results = patents.join(
    same_state_counts,
    patents.PATENT == same_state_counts.CITING,
    "inner"
)

In [23]:
patent_results.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------+--------------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD| CITING|CO_CITED_COUNT|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------+--------------+
|3858245| 1975| 5485|   1972|     US|     NY|  237715|      2|     5|   623|  3|    39|    7|      11|  0.8571| 0.4959|  0.2778|  6.4545|  9.2857|     0.0|     0.0|     0.0|     0.0|3858245|             1|
|3858252| 1975| 5485|   1973|     US|     CA|    NULL|      1|     5|     4|  6|    65|    8|       4|    0.25|  0.375|     0.5|     5.5|   28.75|    NULL|    NULL|    NULL|   

## Final top 10

The final result is sorted by CO_CITED_COUNT in descending order and the
top 10 patents are selected.

In [24]:
top10 = patent_results.orderBy(
    col("CO_CITED_COUNT").desc()
).limit(10)

In [25]:
top10.show(10, truncate=False)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------+--------------+
|PATENT |GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|CITING |CO_CITED_COUNT|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+-------+--------------+
|5959466|1999 |14515|1997   |US     |CA     |5310    |2      |NULL  |326   |4  |46    |159  |0       |1.0     |NULL   |0.6186  |NULL    |4.8868  |0.0455  |0.044   |NULL    |NULL    |5959466|125           |
|5983822|1999 |14564|1998   |US     |TX     |569900  |2      |NULL  |114   |5  |55    |200  |0       |0.995   |NULL   |0.7201  |NULL    |12.45   |0.0     |0.0     |NULL    |NUL